# 32. 합성 평가셋 — 입력 파일 생성

사전 등록: `docs/plans/NLR_EVALSET_PREREGISTRATION.md`

향수에서 자연어 문장을 역생성하기 위한 **업로드용 입력 파일과 프롬프트**를 만든다.
문장 생성은 사람이 GPT 에 직접 올려서 한다. 이 노트북은 API 를 호출하지 않는다.

## 0. 실행 조건과 한계

- **API 호출 0회.** `perfumes.csv` 와 `perfumes.jsonl` 을 읽기만 한다
- 평가 데이터(golden set · 설문)를 읽지도 쓰지도 않는다. 분포 비교는 노트북 33 에서 한다
- `perfumes.jsonl` 509MB 를 한 번 훑는다 (수 분)
- 모집단이 `people >= 500` 이므로 **인기 향수에 치우쳐 있다.** `DECISIONS.md` N4 와 같은 한계다
- 순환은 없앨 수 없고 **크기를 재도록** 두 갈래를 만든다. 사전 등록 §5

In [1]:
import hashlib
import json
import pathlib
import re

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 240)

# True면 계산과 표시만 하고 파일을 만들지 않는다.
REPORT_ONLY = False

SEED = 42
MIN_PEOPLE = 500      # MAP D3 후보 풀 기준
N_PERFUMES = 100      # 표본 향수 수
N_BATCH = 4           # 수작업 업로드 배치 수 (배치마다 계열 균형)
N_SENTENCES = 3       # 향수당 문장 수
TOP_K_C = 3           # 조건 집합 C = strength 상위 3개 accord
TOP_K_A = 5           # 갈래 A 가 LLM 에게 주는 accord 수
PROS_N, CONS_N = 6, 3 # 갈래 B 가 주는 ai_summary 서술문 수 (up_votes 상위)

print(f"REPORT_ONLY: {REPORT_ONLY} / seed {SEED} / 향수 {N_PERFUMES}개 x 문장 {N_SENTENCES}개")

REPORT_ONLY: False / seed 42 / 향수 100개 x 문장 3개


## 1. 경로 · 입력 해싱 · 쓰기 가드

In [2]:
PROJECT_ROOT = pathlib.Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "analysis_outputs"
KNOWLEDGE_DIR = PROJECT_ROOT / "data" / "scent_knowledge"

INPUT_PATHS = {
    "perfumes_csv": PROJECT_ROOT / "perfumes.csv",
    "accord_dict": OUTPUT_DIR / "10_accord_dictionary.csv",
}
JSONL_PATH = PROJECT_ROOT / "perfumes.jsonl"   # 509MB — 해싱하지 않고 존재만 확인

OUTPUT_PATHS = {
    "input_a": OUTPUT_DIR / "32_evalset_input_A.csv",
    "input_b": OUTPUT_DIR / "32_evalset_input_B.csv",
    "prompt_a": OUTPUT_DIR / "32_evalset_prompt_A.txt",
    "prompt_b": OUTPUT_DIR / "32_evalset_prompt_B.txt",
    "answer_key": OUTPUT_DIR / "32_evalset_answer_key.csv",
    "manifest": OUTPUT_DIR / "32_evalset_manifest.json",
    "run_log": OUTPUT_DIR / "32_evalset_run_log.md",
}


def sha256_file(path):
    """파일의 SHA-256 hex digest. str."""
    digest = hashlib.sha256()
    with pathlib.Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_text(text):
    """문자열의 SHA-256 hex digest. str."""
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


missing = [str(p) for p in list(INPUT_PATHS.values()) + [JSONL_PATH] if not p.is_file()]
if missing:
    raise FileNotFoundError(f"필수 입력이 없습니다: {missing}")
input_hashes_before = {k: sha256_file(v) for k, v in INPUT_PATHS.items()}

PROTECTED = {p.resolve() for p in INPUT_PATHS.values()} | {
    JSONL_PATH.resolve(),
    (KNOWLEDGE_DIR / "domain_lexicon_v1.csv").resolve(),
    (KNOWLEDGE_DIR / "domain_lexicon_v1_1.csv").resolve(),
    (PROJECT_ROOT / "evaluation_data" / "stage1"
     / "13_stage1_golden_set_v1_200.xlsx").resolve(),
}
ALLOWED_WRITES = {p.resolve() for p in OUTPUT_PATHS.values()}


def write_output(path, writer):
    """OUTPUT_PATHS 의 경로에만 쓴다. REPORT_ONLY면 생략. pathlib.Path 또는 None."""
    path = pathlib.Path(path).resolve()
    if path not in ALLOWED_WRITES:
        raise RuntimeError(f"쓰기 허용 경로가 아닙니다: {path}")
    if path in PROTECTED:
        raise RuntimeError(f"보호 파일 덮어쓰기 시도: {path.name}")
    if REPORT_ONLY:
        print(f"[REPORT_ONLY] 저장 생략: {path.name}")
        return None
    writer(path)
    print(f"저장: {path.relative_to(PROJECT_ROOT)}")
    return path


display(pd.Series(input_hashes_before, name="sha256").str.slice(0, 16).to_frame())

,sha256
perfumes_csv,cec1ea0b49885303
accord_dict,567e91370e575731


## 2. 사전 등록 — 데이터를 로드하기 전에 고정한다

결과를 보고 바꾸면 사전 등록이 아니게 된다. 전문은 `docs/plans/NLR_EVALSET_PREREGISTRATION.md`.

In [3]:
PREREG = {
    "version": "evalset-v1",
    "population": f"people >= {MIN_PEOPLE}",
    "sampling": "향 계열(description 첫 문장 라벨) 제곱근 비례 할당, 배치마다 라운드로빈 분배",
    "condition_set_C": f"strength 상위 {TOP_K_C}개 accord. 동점이면 accord 이름 오름차순",
    "arm_A_input": f"상위 {TOP_K_A}개 accord + strength (검색 인덱스와 순환)",
    "arm_B_input": f"ai_summary pros {PROS_N} + cons {CONS_N} (up_votes 상위, 사전 근거와 순환)",
    "primary_metric": "Condition Precision@5 — 상위 5개 중 C 를 전부 가진 향수 수 / 5",
    "secondary_metrics": [
        "느슨한 정의: C 중 2개 이상",
        "원본 향수 Recall@5 · MRR (랜덤 기대값 5/|C만족| 과 병기)",
        "갈래 A 점수 − 갈래 B 점수 = 검색 인덱스 직결 순환의 크기",
    ],
    "diagnostics": ["5위 동점 수", "조건 0개 문장 비율", "완화 단계 발동 횟수",
                    "설문 155건과의 분포 비교 (노트북 33)"],
    "regeneration_rule": "형식 오류와 규칙 2·3 위반만 재생성. 문장이 마음에 안 든다는 이유로는 재생성하지 않는다",
    "not_allowed": ["합격선을 이 노트북에서 정하지 않는다",
                    "평가 데이터를 수정하지 않는다",
                    "노트북 13·14·15·16 을 실행하거나 수정하지 않는다"],
}
display(pd.Series({k: (v if isinstance(v, str) else " / ".join(v))
                   for k, v in PREREG.items()}, name="사전 등록").to_frame())

,사전 등록
version,evalset-v1
population,people >= 500
sampling,"향 계열(description 첫 문장 라벨) 제곱근 비례 할당, 배치마다 라운드로빈 분배"
condition_set_C,strength 상위 3개 accord. 동점이면 accord 이름 오름차순
arm_A_input,상위 5개 accord + strength (검색 인덱스와 순환)
arm_B_input,"ai_summary pros 6 + cons 3 (up_votes 상위, 사전 근거와 순환)"
primary_metric,Condition Precision@5 — 상위 5개 중 C 를 전부 가진 향수 수 / 5
secondary_metrics,느슨한 정의: C 중 2개 이상 / 원본 향수 Recall@5 · MRR (랜덤 기대값 5/|C만족|...
diagnostics,5위 동점 수 / 조건 0개 문장 비율 / 완화 단계 발동 횟수 / 설문 155건과의 분포 비교 (노...
regeneration_rule,형식 오류와 규칙 2·3 위반만 재생성. 문장이 마음에 안 든다는 이유로는 재생성하지 않는다


## 3. 재현 게이트 — 원자료가 기존 측정과 같은지 먼저 확인한다

In [4]:
usecols = ["id", "name", "brand", "people", "accords", "description"]
df = pd.read_csv(INPUT_PATHS["perfumes_csv"], usecols=usecols, low_memory=False)
print(f"perfumes.csv {len(df):,}행")


def parse_accords(value):
    """'name:strength|...' 을 [(name, strength)] 로. list[tuple[str, float]]."""
    out = []
    for part in str(value).split("|"):
        if not part or part == "nan":
            continue
        name, _, strength = part.partition(":")
        try:
            out.append((name, float(strength)))
        except ValueError:
            continue
    out.sort(key=lambda x: (-x[1], x[0]))     # 결정적 tie-break
    return out


df["acc"] = df["accords"].map(parse_accords)

counts = {}
for pairs in df["acc"]:
    for name, _ in pairs:
        counts[name] = counts.get(name, 0) + 1
ad = pd.read_csv(INPUT_PATHS["accord_dict"])
gate_err = sum(counts.get(r["accord"], 0) != r["perfume_count"] for r in ad.to_dict("records"))
print(f"[게이트] accord {len(ad)}개 perfume_count 대조 → 오차 {gate_err}건")
if gate_err:
    raise RuntimeError("재현 게이트 실패: 원자료가 기존 측정과 다르다")

perfumes.csv 131,930행


[게이트] accord 92개 perfume_count 대조 → 오차 0건


## 4. 후보 풀과 향 계열

계열은 `description` 첫 문장의 라벨을 쓴다 (측정 기록 5번). 새 계열 체계를 만들지 않는다.

In [5]:
FAMILY_PAT = re.compile(
    r"\bis an? ((?:[A-Z][a-z]+ ){0,3}[A-Z][a-z]+) fragrance for (?:women|men)")


def family_of(text):
    """description 첫 문장의 향 계열 라벨. str 또는 None."""
    m = FAMILY_PAT.search(str(text))
    return m.group(1) if m else None


pool = df[df["people"].fillna(0) >= MIN_PEOPLE].copy()
pool["family"] = pool["description"].map(family_of)
pool = pool[pool["acc"].map(len) >= TOP_K_A]          # 갈래 A 입력을 채울 수 있어야 한다

print(f"후보 풀 (people >= {MIN_PEOPLE}, accord {TOP_K_A}개 이상)  {len(pool):,}개")
print(f"  계열 라벨 보유  {pool['family'].notna().sum():,} ({pool['family'].notna().mean():.1%})")
pool = pool[pool["family"].notna()].copy()
display(pool["family"].value_counts().head(12).to_frame("후보 수"))

후보 풀 (people >= 500, accord 5개 이상)  7,780개
  계열 라벨 보유  7,272 (93.5%)


,후보 수
family,
Oriental Floral,864
Floral Fruity,716
Floral,706
Oriental Woody,531
Floral Woody Musk,438
Oriental Vanilla,414
Woody Aromatic,406
Oriental,290
Oriental Spicy,279


### `ai_summary` 보유로 한 번 더 거른다

갈래 A 와 갈래 B 가 **같은 향수 집합**을 써야 `A − B` 차이를 순환 크기로 읽을 수 있다.
한쪽에만 있는 향수가 섞이면 그 차이가 표본 차이인지 순환 차이인지 구분되지 않는다.

그래서 표집하기 **전에** `perfumes.jsonl` 을 훑어 `ai_summary` 를 가진 향수만 남긴다.

In [6]:
pool_ids = set(pool["id"].astype(str))
summaries = {}
scanned = 0
with JSONL_PATH.open("r", encoding="utf-8") as handle:
    for line in handle:
        if '"ai_summary"' not in line:
            continue
        try:
            obj = json.loads(line)
        except ValueError:
            continue
        pid = str(obj.get("id"))
        if pid not in pool_ids:
            continue
        scanned += 1
        s = obj.get("ai_summary") or {}

        def top(items, k):
            items = [i for i in (items or []) if (i or {}).get("text")]
            items.sort(key=lambda i: -(i.get("up_votes") or 0))
            return [i["text"].strip() for i in items[:k]]

        pros, cons = top(s.get("pros"), PROS_N), top(s.get("cons"), CONS_N)
        if pros or cons:
            summaries[pid] = {"pros": pros, "cons": cons}

print(f"후보 풀 {len(pool):,}개 중 jsonl 에서 본 것 {scanned:,}개")
print(f"  ai_summary 서술문 보유  {len(summaries):,}개 ({len(summaries)/len(pool):.1%})")

pool = pool[pool["id"].astype(str).isin(summaries)].copy()
print(f"최종 후보 풀  {len(pool):,}개 / 계열 {pool['family'].nunique()}종")

후보 풀 7,272개 중 jsonl 에서 본 것 7,272개
  ai_summary 서술문 보유  6,883개 (94.7%)
최종 후보 풀  6,883개 / 계열 32종


## 5. 표집 — 계열 제곱근 비례, 배치마다 균형

MAP `D3` 이 지도 향수를 고를 때 쓴 제곱근 비례 할당을 그대로 쓴다.
수작업 업로드이므로 **배치마다 독립적으로 균형**을 맞춰, 일부 배치만 해도 계열이 치우치지 않게 한다.

In [7]:
import math

fam_counts = pool["family"].value_counts()
weights = fam_counts.map(math.sqrt)
quota = (weights / weights.sum() * N_PERFUMES).round().astype(int)
quota = quota.clip(upper=fam_counts)                  # 후보보다 많이 뽑지 않는다

# 합을 정확히 N_PERFUMES 로 맞춘다 (큰 계열부터 가감)
for _ in range(1000):
    if quota.sum() == N_PERFUMES:
        break
    diff = N_PERFUMES - quota.sum()
    order = quota.sort_values(ascending=(diff > 0)).index
    for fam in order:
        if diff > 0 and quota[fam] < fam_counts[fam]:
            quota[fam] += 1
            diff -= 1
        elif diff < 0 and quota[fam] > 0:
            quota[fam] -= 1
            diff += 1
        if diff == 0:
            break

picked = []
for fam, n in quota[quota > 0].items():
    picked.append(pool[pool["family"] == fam].sample(n=int(n), random_state=SEED))
sample = pd.concat(picked)

# 계열 안에서 섞은 뒤 라운드로빈으로 배치에 나눠 담는다 → 배치마다 균형
sample = sample.sample(frac=1.0, random_state=SEED).sort_values("family", kind="stable")
sample["batch"] = [(i % N_BATCH) + 1 for i in range(len(sample))]
sample = sample.sort_values(["batch", "family", "id"]).reset_index(drop=True)

print(f"표본 {len(sample)}개 / 계열 {sample['family'].nunique()}종")
display(pd.crosstab(sample["family"], sample["batch"]).assign(
    합계=lambda t: t.sum(axis=1)).sort_values("합계", ascending=False).head(12))

표본 100개 / 계열 32종


batch,1,2,3,4,합계
family,,,,,
Oriental Floral,2,2,1,1,6
Floral,1,1,1,2,5
Floral Woody Musk,2,1,1,1,5
Floral Fruity,2,1,1,1,5
Oriental Vanilla,2,1,1,1,5
Oriental Woody,1,2,1,1,5
Woody Aromatic,1,1,1,2,5
Oriental Spicy,1,1,1,1,4
Floral Fruity Gourmand,1,1,1,1,4


## 6. 조건 집합 C 와 랜덤 기대값

In [8]:
sample["C"] = sample["acc"].map(lambda ps: [n for n, _ in ps[:TOP_K_C]])

# 전체 코퍼스에서 C 를 전부 가진 향수 수 — 랜덤 기대값의 분모
all_sets = df["acc"].map(lambda ps: {n for n, _ in ps}).tolist()


def count_all_of(targets):
    """targets 를 전부 가진 향수 수. int."""
    t = set(targets)
    return sum(1 for s in all_sets if t <= s)


sample["n_match_all"] = sample["C"].map(count_all_of)
sample["random_expect_at5"] = 5.0 / sample["n_match_all"].clip(lower=5)

display(sample["n_match_all"].describe()[["min", "25%", "50%", "75%", "max"]].to_frame("C 만족 향수 수"))
print(f"랜덤 기대값(원본이 상위5에 들 확률) 중앙값  {sample['random_expect_at5'].median():.4f}")
print(f"C 만족 향수가 5개 미만인 표본  {(sample['n_match_all'] < 5).sum()}개")

,C 만족 향수 수
min,2.0
25%,1608.0
50%,6098.0
75%,11958.0
max,24708.0


랜덤 기대값(원본이 상위5에 들 확률) 중앙값  0.0008
C 만족 향수가 5개 미만인 표본  1개


## 7. 갈래 A 입력 — 상위 accord

향수 이름·브랜드를 넣지 않는다. 업로드 파일에 이름이 있으면 LLM 이 그것을 문장에 쓴다.

In [9]:
input_a = pd.DataFrame({
    "batch": sample["batch"],
    "perfume_id": sample["id"],
    "accords": sample["acc"].map(
        lambda ps: ", ".join(f"{n}({int(v)})" for n, v in ps[:TOP_K_A])),
})
display(input_a.head(5))

,batch,perfume_id,accords
0,1,17140,"coconut(100), sweet(40), lactonic(35), tropical(30), van..."
1,1,103715,"aromatic(100), fruity(63), marine(58), woody(56), tropic..."
2,1,5581,"woody(100), warm spicy(68), fruity(65), sweet(43), amber..."
3,1,30544,"citrus(100), aromatic(66), green(54), fresh spicy(45), l..."
4,1,87922,"tobacco(100), musky(88), woody(83), rose(78), patchouli(67)"


## 8. 갈래 B 입력 — `ai_summary`

앞에서 모아둔 서술문을 표본 순서대로 편다. 표집 전에 걸렀으므로 **갈래 A 와 행 수가 같다.**

In [10]:
rows = []
for r in sample.to_dict("records"):
    s = summaries[str(r["id"])]
    text = " / ".join(s["pros"])
    if s["cons"]:
        text += "  [단점] " + " / ".join(s["cons"])
    rows.append({"batch": r["batch"], "perfume_id": r["id"], "opinions": text})
input_b = pd.DataFrame(rows)
print(f"갈래 B 입력 {len(input_b)}행 (갈래 A {len(input_a)}행)")
display(input_b.head(3))

갈래 B 입력 100행 (갈래 A 100행)


,batch,perfume_id,opinions
0,1,17140,Suitable for daytime wear / Uplifting summer scent / Sim...
1,1,103715,"Friendly entry-level fragrance for beginners / Modern, a..."
2,1,5581,Recommended for those who want something different / Sui...


## 9. 검증

In [11]:
problems = []

if len(sample) != N_PERFUMES:
    problems.append(f"표본 수가 {len(sample)}개 (기대 {N_PERFUMES})")
if sample["id"].duplicated().any():
    problems.append("표본에 중복 id 가 있다")
if sample["C"].map(len).ne(TOP_K_C).any():
    problems.append("조건 집합 C 의 크기가 3이 아닌 행이 있다")
if input_a["accords"].str.strip().eq("").any():
    problems.append("갈래 A 입력에 빈 accord 가 있다")

if len(input_b) != len(input_a):
    problems.append(f"갈래 A({len(input_a)})와 B({len(input_b)})의 행 수가 다르다")
if quota.sum() != N_PERFUMES:
    problems.append(f"계열 할당 합이 {int(quota.sum())} (기대 {N_PERFUMES})")

# 배치 균형
spread = pd.crosstab(sample["family"], sample["batch"]).max(axis=1) - \
         pd.crosstab(sample["family"], sample["batch"]).min(axis=1)
if spread.max() > 1:
    problems.append(f"배치 간 계열 편차가 1을 넘는다 (최대 {spread.max()})")

print("검증 통과" if not problems else "검증 실패")
for p in problems:
    print(" -", p)

검증 통과


### 경고 — 갈래 B 본문에 향수를 특정할 단서가 있는가

`Coconut Passion` 의 요약에 `coconut` 이 나오는 것은 누출이 아니라 **그 향수가 실제로
코코넛 향이기 때문**이다. 반면 `Chanel` 같은 토큰은 향수를 특정한다.

둘을 가르려고 **코퍼스 빈도**를 쓴다. 후보 풀 전체 서술문의 1% 이상에 나오는 토큰은
보통 단어로 보고 넘긴다. 차단하지 않고 목록만 남긴다 — 규칙 3의 실제 집행은
**생성된 문장** 쪽에서 한다(노트북 33).

In [12]:
pool_texts = [" / ".join(s["pros"] + s["cons"]).lower() for s in summaries.values()]
COMMON_SHARE = 0.01


def doc_share(token):
    """후보 풀 서술문 중 이 토큰이 나오는 비율. float."""
    pat = re.compile(rf"\b{re.escape(token)}\b")
    return sum(1 for t in pool_texts if pat.search(t)) / len(pool_texts)


own_text = dict(zip(input_b["perfume_id"], input_b["opinions"].str.lower()))
hits = []
for r in sample.to_dict("records"):
    text = own_text.get(r["id"], "")
    for tok in set(re.split(r"[^A-Za-z]+", f"{r['name']} {r['brand']}")):
        low = tok.lower()
        if len(low) < 4 or not re.search(rf"\b{re.escape(low)}\b", text):
            continue
        share = doc_share(low)
        hits.append({"perfume_id": r["id"], "token": tok,
                     "코퍼스 등장 비율": round(share, 4),
                     "판정": "보통 단어" if share >= COMMON_SHARE else "특정 가능"})

leak_df = pd.DataFrame(hits)
if len(leak_df):
    n_id = int((leak_df["판정"] == "특정 가능").sum())
    print(f"본문에 자기 이름 토큰이 나온 경우 {len(leak_df)}건 / 그중 특정 가능 {n_id}건")
    display(leak_df.sort_values("코퍼스 등장 비율").head(12))
else:
    n_id = 0
    print("본문에 자기 이름 토큰이 나온 경우 없음")

본문에 자기 이름 토큰이 나온 경우 20건 / 그중 특정 가능 9건


,perfume_id,token,코퍼스 등장 비율,판정
6,13737,Saint,0.0001,특정 가능
7,13737,Laurent,0.0001,특정 가능
13,66933,Natura,0.0006,특정 가능
5,13737,Yves,0.0009,특정 가능
2,85038,Hypnotic,0.0026,특정 가능
15,8700,Blood,0.0032,특정 가능
14,8700,Basil,0.0038,특정 가능
17,11560,Heliotrope,0.0046,특정 가능
11,80042,Marine,0.0096,특정 가능
4,3247,Relaxing,0.0128,보통 단어


## 10. 프롬프트

In [13]:
Q = chr(34)      # 따옴표를 문자열 안에 중첩하지 않으려고 코드로 만든다

COMMON_RULES = "\n".join([
    "[규칙]",
    "1. 한국어로 쓴다.",
    "2. 영어 향 용어(soapy, woody, musky 등)를 문장에 그대로 쓰지 않는다.",
    "   한국 사람이 일상에서 쓰는 말로 바꿔 쓴다.",
    "3. 향수 이름·브랜드·조향사·출시 연도를 쓰지 않는다.",
    "4. 한 문장은 70자 내외로 하고 조건을 2~3개 담는다.",
    f"   예: {Q}여름에 쓸 건데 빨래 냄새 나고 이불 같은 느낌이면 좋겠어. 머스크는 빼줘.{Q}",
    f"5. 한 향수의 문장 {N_SENTENCES}개는 서로 달라야 한다. 같은 표현을 반복하지 않는다.",
    "6. 해설·머리말·맺음말을 붙이지 말고 아래 CSV 만 출력한다.",
    "",
    "[출력 형식] 헤더를 포함한 CSV. 문장은 큰따옴표로 감싼다.",
    "perfume_id,sentence_no,sentence",
] + [f"12345,{i},{Q}...{Q}" for i in range(1, N_SENTENCES + 1)])

HEAD = "향수 추천 시스템의 평가용 문장을 만든다."
ASK = ("각 향수마다, 그 향수를 찾고 싶어하는 한국 사용자가 검색창에 쓸 법한 문장을 "
       f"{N_SENTENCES}개씩 만들어라.")

PROMPT_A = "\n".join([
    HEAD, "",
    "아래 표는 향수별 대표 향 특성과 그 세기(0~100)다.",
    ASK, "",
    COMMON_RULES, "",
    "[입력] batch, perfume_id, accords",
    "아래에 32_evalset_input_A.csv 의 batch 한 개 분량을 붙여넣는다.",
    "",
])

PROMPT_B = "\n".join([
    HEAD, "",
    "아래 표는 향수별 사용자 의견 요약이다.",
    "향에 대한 서술만 참고하고, 지속력·가격·포장·병 디자인처럼 향과 무관한 내용은 무시한다.",
    ASK, "",
    COMMON_RULES, "",
    "[입력] batch, perfume_id, opinions",
    "아래에 32_evalset_input_B.csv 의 batch 한 개 분량을 붙여넣는다.",
    "",
])

print(PROMPT_A)

향수 추천 시스템의 평가용 문장을 만든다.

아래 표는 향수별 대표 향 특성과 그 세기(0~100)다.
각 향수마다, 그 향수를 찾고 싶어하는 한국 사용자가 검색창에 쓸 법한 문장을 3개씩 만들어라.

[규칙]
1. 한국어로 쓴다.
2. 영어 향 용어(soapy, woody, musky 등)를 문장에 그대로 쓰지 않는다.
   한국 사람이 일상에서 쓰는 말로 바꿔 쓴다.
3. 향수 이름·브랜드·조향사·출시 연도를 쓰지 않는다.
4. 한 문장은 70자 내외로 하고 조건을 2~3개 담는다.
   예: "여름에 쓸 건데 빨래 냄새 나고 이불 같은 느낌이면 좋겠어. 머스크는 빼줘."
5. 한 향수의 문장 3개는 서로 달라야 한다. 같은 표현을 반복하지 않는다.
6. 해설·머리말·맺음말을 붙이지 말고 아래 CSV 만 출력한다.

[출력 형식] 헤더를 포함한 CSV. 문장은 큰따옴표로 감싼다.
perfume_id,sentence_no,sentence
12345,1,"..."
12345,2,"..."
12345,3,"..."

[입력] batch, perfume_id, accords
아래에 32_evalset_input_A.csv 의 batch 한 개 분량을 붙여넣는다.



## 11. 저장

In [14]:
answer_key = pd.DataFrame({
    "perfume_id": sample["id"],
    "family": sample["family"],
    "batch": sample["batch"],
    "C": sample["C"].map(lambda c: "|".join(c)),
    "n_match_all": sample["n_match_all"],
    "random_expect_at5": sample["random_expect_at5"].round(6),
})

RUN_LOG = """# 32. 합성 평가셋 — 생성 실행 기록

사전 등록: `docs/plans/NLR_EVALSET_PREREGISTRATION.md` §6

**생성한 사람이 채운다.** 모델과 온도를 통제할 수 없는 것이 이 방식의 한계이므로
무엇으로 돌렸는지 남겨야 결과를 읽을 수 있다.

| 항목 | 값 |
|---|---|
| 사용한 모델 이름·버전 | |
| 생성 일시 | |
| 프롬프트를 그대로 썼는가 | |
| 고쳤다면 무엇을 | |

## 배치별 기록

| 갈래 | 배치 | 재생성 횟수 | 사유 |
|---|---|---|---|
| A | 1 | | |
| A | 2 | | |
| A | 3 | | |
| A | 4 | | |
| B | 1 | | |
| B | 2 | | |
| B | 3 | | |
| B | 4 | | |

재생성은 **형식 오류와 규칙 2·3 위반만** 허용한다.
문장이 마음에 안 든다는 이유로 다시 돌리면 사전 등록이 깨진다.

## 검수 (갈래별 15건)

| 갈래 | 본 건수 | 한국어가 어색 | 규칙 위반 | 향수 설명으로 안 읽힘 |
|---|---|---|---|---|
| A | | | | |
| B | | | | |
"""

manifest = {
    "prereg": PREREG,
    "seed": SEED,
    "n_perfumes": int(len(sample)),
    "n_batch": N_BATCH,
    "n_sentences": N_SENTENCES,
    "population": (f"people >= {MIN_PEOPLE} and len(accords) >= {TOP_K_A} "
                   "and family is not None and ai_summary 서술문 보유"),
    "pool_size_after_ai_summary_filter": int(len(pool)),
    "ai_summary_holders_in_pool": int(len(summaries)),
    "input_sha256": input_hashes_before,
    "jsonl_size_bytes": JSONL_PATH.stat().st_size,
    "prompt_a_sha256": sha256_text(PROMPT_A),
    "prompt_b_sha256": sha256_text(PROMPT_B),
    "family_quota": {k: int(v) for k, v in quota[quota > 0].items()},
    "sampled_ids_by_batch": {
        str(b): sorted(int(i) for i in g["id"]) for b, g in sample.groupby("batch")},
    "arm_a_rows": int(len(input_a)),
    "arm_b_rows": int(len(input_b)),
    "arm_b_identifiable_tokens": (
        leak_df[leak_df["판정"] == "특정 가능"].to_dict("records") if len(leak_df) else []),
    "validation_problems": problems,
}

write_output(OUTPUT_PATHS["input_a"], lambda p: input_a.to_csv(p, index=False, encoding="utf-8-sig"))
write_output(OUTPUT_PATHS["input_b"], lambda p: input_b.to_csv(p, index=False, encoding="utf-8-sig"))
write_output(OUTPUT_PATHS["answer_key"], lambda p: answer_key.to_csv(p, index=False, encoding="utf-8-sig"))
write_output(OUTPUT_PATHS["prompt_a"], lambda p: p.write_text(PROMPT_A, encoding="utf-8"))
write_output(OUTPUT_PATHS["prompt_b"], lambda p: p.write_text(PROMPT_B, encoding="utf-8"))
write_output(OUTPUT_PATHS["run_log"], lambda p: p.write_text(RUN_LOG, encoding="utf-8"))
write_output(OUTPUT_PATHS["manifest"],
             lambda p: p.write_text(json.dumps(manifest, ensure_ascii=False, indent=2),
                                    encoding="utf-8"))

저장: analysis_outputs\32_evalset_input_A.csv
저장: analysis_outputs\32_evalset_input_B.csv
저장: analysis_outputs\32_evalset_answer_key.csv
저장: analysis_outputs\32_evalset_prompt_A.txt
저장: analysis_outputs\32_evalset_prompt_B.txt
저장: analysis_outputs\32_evalset_run_log.md
저장: analysis_outputs\32_evalset_manifest.json


WindowsPath('C:/Users/SSAFY/Desktop/hyanghae/EDA/analysis_outputs/32_evalset_manifest.json')

## 12. 가드 검증 — 입력이 안 바뀌었는지

In [15]:
after = {k: sha256_file(v) for k, v in INPUT_PATHS.items()}
changed = [k for k in after if after[k] != input_hashes_before[k]]
if changed:
    raise RuntimeError(f"입력 파일이 변경됐다: {changed}")
print("입력 파일 해시 불변 확인:", ", ".join(INPUT_PATHS))
print()
print("다음 할 일 — analysis_outputs/32_evalset_prompt_A.txt 와 input_A.csv 의 batch 1 을")
print("GPT 에 올려 문장을 만들고, 받은 CSV 를 evaluation_data/nlr_evalset/ 아래 저장한다.")
print("기록은 32_evalset_run_log.md 에 남긴다.")

입력 파일 해시 불변 확인: perfumes_csv, accord_dict

다음 할 일 — analysis_outputs/32_evalset_prompt_A.txt 와 input_A.csv 의 batch 1 을
GPT 에 올려 문장을 만들고, 받은 CSV 를 evaluation_data/nlr_evalset/ 아래 저장한다.
기록은 32_evalset_run_log.md 에 남긴다.
